In [ ]:
# Cell 1: Check GPU, CUDA & cuDNN versions
import os
import subprocess

print("=" * 50)
print("GPU Info:")
print("=" * 50)
os.system("nvidia-smi")

print("\n" + "=" * 50)
print("CUDA & cuDNN Check:")
print("=" * 50)

# Check CUDA toolkit version
os.system("nvcc --version")

# Check cuDNN from ldconfig
result = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True)
cudnn_lines = [l for l in result.stdout.split("\n") if "cudnn" in l.lower()]
if cudnn_lines:
    print("\ncuDNN libraries found:")
    for l in cudnn_lines[:5]:
        print(f"  {l.strip()}")
else:
    print("\n⚠️ No cuDNN found in ldconfig — will install below")

## Install Dependencies
Install PaddlePaddle (GPU) + PaddleOCR 3.x with the correct CUDA/cuDNN for the Colab T4 runtime.

In [1]:

import subprocess, re, sys, os, importlib.util, ctypes, glob, site, shutil

# Set env vars BEFORE any paddle imports
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# ── Pre-load newer NCCL BEFORE paddle is imported ────────────────────────────
# paddle's bundled libnccl.so.2 is old (missing ncclCommShrink, added NCCL 2.21.5).
# Loading newer NCCL first → its SONAME takes the SO cache slot → paddle's
# dlopen returns our handle. ncclCommShrink available → no ImportError later.
_nccl_paths = []
for _sp in site.getsitepackages():
    _nccl_paths += glob.glob(os.path.join(_sp, "nvidia", "nccl", "lib", "libnccl.so.2*"))
if not _nccl_paths:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "nvidia-nccl-cu12"])
    for _sp in site.getsitepackages():
        _nccl_paths += glob.glob(os.path.join(_sp, "nvidia", "nccl", "lib", "libnccl.so.2*"))
for _p in sorted(set(_nccl_paths), reverse=True):
    try:
        ctypes.CDLL(_p, os.RTLD_GLOBAL)
        print(f"✅ NCCL pre-loaded: {_p}")
        break
    except OSError:
        pass
else:
    print("⚠️ Could not pre-load new NCCL (will try to inject in OCR Engine cell)")

_NEED_INSTALL = True

# ── Check if paddle 3.x with GPU is already active ───────────────────────────
try:
    import paddle as _pc
    _ver = _pc.__version__
    _gpu = _pc.device.is_compiled_with_cuda()
    if _ver.startswith("3") and _gpu:
        print(f"✅ PaddlePaddle {_ver} (GPU) already loaded")
        print(f"   GPU count: {_pc.device.cuda.device_count()}")
        if _pc.device.cuda.device_count() > 0:
            print(f"   GPU name : {_pc.device.cuda.get_device_name(0)}")
        # Check if paddleocr EXISTS (without importing it — avoids PDX init here)
        if importlib.util.find_spec("paddleocr") is not None:
            print(f"✅ paddleocr package found")
        else:
            print("\n→ Installing paddleocr>=3.0.0 ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddleocr>=3.0.0"], check=True)
            print("✅ paddleocr installed")
        _NEED_INSTALL = False
    else:
        print(f"⚠️ Paddle {_ver} loaded but CUDA={_gpu} — need reinstall")
except ImportError:
    print("paddle not installed yet")
except Exception as e:
    print(f"paddle check failed: {e}")

# ── Install section (only runs if needed) ─────────────────────────────────────
if _NEED_INSTALL:
    try:
        nvcc_out = subprocess.check_output(["nvcc", "--version"], text=True)
        cuda_ver = re.search(r"release (\d+\.\d+)", nvcc_out).group(1)
    except Exception:
        cuda_ver = "12.8"
    py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
    print(f"CUDA: {cuda_ver} | Python: {py_ver}")

    # ── Uninstall ALL existing paddle packages first ──────────────────────────
    print("\n→ Removing ALL existing paddle packages...")
    for pkg in ["paddlepaddle-gpu", "paddlepaddle", "paddleocr"]:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", pkg], capture_output=True)
    for sp in site.getsitepackages():
        for d in ["paddle", "paddlepaddle", "paddlepaddle_gpu"]:
            target = os.path.join(sp, d)
            if os.path.isdir(target):
                shutil.rmtree(target, ignore_errors=True)
                print(f"  Removed leftover: {target}")
    print("  ✅ Clean slate")

    # ── Direct CDN wheel URLs ─────────────────────────────────────────────────
    WHEEL_URLS = {
        "cu128": {
            "cp310": "https://paddle-whl.bj.bcebos.com/stable/cu128/paddlepaddle-gpu/paddlepaddle-3.2.0-cp310-cp310-linux_x86_64.whl",
            "cp311": "https://paddle-whl.bj.bcebos.com/stable/cu128/paddlepaddle-gpu/paddlepaddle-3.2.0-cp311-cp311-linux_x86_64.whl",
            "cp312": "https://paddle-whl.bj.bcebos.com/stable/cu128/paddlepaddle-gpu/paddlepaddle-3.2.0-cp312-cp312-linux_x86_64.whl",
            "cp313": "https://paddle-whl.bj.bcebos.com/stable/cu128/paddlepaddle-gpu/paddlepaddle-3.2.0-cp313-cp313-linux_x86_64.whl",
        },
        "cu126": {
            "cp310": "https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.0.0-cp310-cp310-linux_x86_64.whl",
            "cp311": "https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.0.0-cp311-cp311-linux_x86_64.whl",
            "cp312": "https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.0.0-cp312-cp312-linux_x86_64.whl",
        },
    }
    if cuda_ver.startswith("12"):
        minor = int(cuda_ver.split(".")[1])
        buckets = ["cu128", "cu126"] if minor >= 8 else ["cu126"]
    else:
        buckets = ["cu128", "cu126"]

    installed = False
    for bucket in buckets:
        url = WHEEL_URLS.get(bucket, {}).get(py_ver)
        if not url:
            continue
        print(f"\n→ Installing from {bucket}...")
        result = subprocess.run([sys.executable, "-m", "pip", "install", url], capture_output=True, text=True)
        if result.returncode == 0:
            ver_check = subprocess.run(
                [sys.executable, "-c",
                 "import paddle; print(paddle.__version__, paddle.device.is_compiled_with_cuda())"],
                capture_output=True, text=True
            )
            out = ver_check.stdout.strip()
            print(f"  Subprocess says: {out}")
            if "3." in out and "True" in out:
                installed = True
                break
            elif "3." in out:
                print(f"  ⚠️ CUDA=False — trying next bucket")
        else:
            err = result.stderr.strip().split("\n")
            print(f"  ✗ {err[-1] if err else 'unknown'}")

    if not installed:
        raise RuntimeError(f"Could not install PaddlePaddle 3.x GPU for {py_ver} / CUDA {cuda_ver}")

    # Install PaddleOCR
    print("\n→ Installing paddleocr>=3.0.0 ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddleocr>=3.0.0"], check=True)
    print("✅ paddleocr installed")

    print("\n" + "="*60)
    print("✅ Install complete! Restarting kernel...")
    print("→ After restart, re-run this cell (it will NOT reinstall)")
    print("="*60)
    os.kill(os.getpid(), 9)


✅ NCCL pre-loaded: /usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib/libnccl.so.2


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


✅ PaddlePaddle 3.0.0 (GPU) already loaded
   GPU count: 1
   GPU name : Tesla T4
✅ paddleocr package found


## Mount Google Drive & Setup Paths

### How to get images into Colab (reliable method)
Uploading 150K individual files to Drive will always crash. Instead:

1. **Locally**, zip the images folder (run once in your terminal):
   ```
   cd dataset
   zip -r img_resized.zip img_resized/
   ```
   On Windows PowerShell:
   ```
   Compress-Archive -Path dataset\img_resized -DestinationPath dataset\img_resized.zip
   ```
2. **Upload** the single `img_resized.zip` file to your Google Drive (one file = reliable)
3. **Run the cell below** — it mounts Drive, copies the zip to Colab's local SSD, and unzips it there

Images are read from Colab's fast local SSD (`/content/img_resized/`), outputs are saved back to Drive.

In [11]:

from google.colab import drive
from pathlib import Path
import os, shutil, zipfile, time

drive.mount("/content/drive")

# ── Adjust this path to match your Google Drive folder structure ──
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/EndSem_Project")

# Output dirs (on Drive — persists after session ends)
OCR_OUTPUT_DIR   = DRIVE_PROJECT_ROOT / "dataset" / "img_txt_collab"
OCR_CONSOLIDATED = DRIVE_PROJECT_ROOT / "dataset" / "ocr_consolidated_collab.json"
OCR_FAILURES     = DRIVE_PROJECT_ROOT / "dataset" / "ocr_failures_collab.txt"
OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Restore previous OCR results from img_txt_collab.zip (if present) ────────
# If you zipped up img_txt_collab/ from a prior session and uploaded it to Drive,
# this unzips it back to OCR_OUTPUT_DIR so the OCR loop resumes correctly.
PREV_OCR_ZIP = DRIVE_PROJECT_ROOT / "dataset" / "img_txt_collab.zip"
if PREV_OCR_ZIP.exists():
    already_restored = len(list(OCR_OUTPUT_DIR.glob("*.json")))
    if already_restored > 0:
        print(f"✅ OCR results already present ({already_restored} files) — skipping unzip of img_txt_collab.zip")
    else:
        print(f"→ Unzipping img_txt_collab.zip → {OCR_OUTPUT_DIR} ...")
        t0 = time.time()
        with zipfile.ZipFile(str(PREV_OCR_ZIP), "r") as zf:
            members = zf.namelist()
            # Support both flat zips and zips with a top-level folder
            for member in members:
                if not member.endswith(".json"):
                    continue
                filename = Path(member).name  # strip any leading folder
                target = OCR_OUTPUT_DIR / filename
                with zf.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
        restored = len(list(OCR_OUTPUT_DIR.glob("*.json")))
        print(f"  ✅ Restored {restored} OCR files in {time.time()-t0:.0f}s")
else:
    print("ℹ️  No img_txt_collab.zip found on Drive — starting fresh or continuing from existing files")

# ── Unzip archive.zip to Colab local SSD (fast reads, avoids Drive I/O during OCR) ──
# archive.zip structure: img_resized/, img_txt/, splits/, MMHS150K_GT.json, etc.
LOCAL_IMG_DIR = Path("/content/img_resized")
ZIP_ON_DRIVE  = DRIVE_PROJECT_ROOT / "dataset" / "archive.zip"

if LOCAL_IMG_DIR.exists() and len(list(LOCAL_IMG_DIR.glob("*.jpg"))) > 0:
    print(f"✅ img_resized already unzipped locally ({len(list(LOCAL_IMG_DIR.glob('*.jpg')))} images)")
else:
    if not ZIP_ON_DRIVE.exists():
        raise FileNotFoundError(
            f"Zip not found at {ZIP_ON_DRIVE}\n"
            "Please upload archive.zip to: MyDrive/EndSem_Project/dataset/archive.zip"
        )
    print("Copying archive.zip from Drive to local SSD...")
    t0 = time.time()
    shutil.copy(str(ZIP_ON_DRIVE), "/content/archive.zip")
    print(f"  Copy done in {time.time()-t0:.0f}s, extracting...")
    t0 = time.time()
    with zipfile.ZipFile("/content/archive.zip", "r") as zf:
        zf.extractall("/content/")
    os.remove("/content/archive.zip")  # free up space
    print(f"  Extracted in {time.time()-t0:.0f}s ✅")

# Use local SSD for image reads
IMG_DIR = LOCAL_IMG_DIR

# Sanity check
img_count    = len(list(IMG_DIR.glob("*.jpg")))
already_done = len(list(OCR_OUTPUT_DIR.glob("*.json")))
print(f"\nImages on local SSD: {img_count}")
print(f"Already processed in img_txt_collab (Drive): {already_done}")
print(f"Remaining: {img_count - already_done}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ OCR results already present (108898 files) — skipping unzip of img_txt_collab.zip
✅ img_resized already unzipped locally (150000 images)

Images on local SSD: 150000
Already processed in img_txt_collab (Drive): 108898
Remaining: 41102


## OCR Engine (PaddleOCR 3.x / PP-OCRv5)
Same engine as the local `preprocess_ocr.py` script, with EasyOCR fallback.

In [12]:

import os, sys, site, re, subprocess, ctypes, glob

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# ── Fix 1: Inject new NCCL symbols into live process ─────────────────────────
# paddle's old bundled libnccl.so.2 (loaded with RTLD_GLOBAL) is missing
# ncclCommShrink (added in NCCL 2.21.5). libtorch_cuda.so needs it.
# Loading a NEWER nccl by full path with RTLD_GLOBAL adds the missing symbol
# to the global table — existing old symbols stay, new ones get added.
print("→ Injecting new NCCL symbols into process...")
_nccl_paths = []
for sp in site.getsitepackages():
    _nccl_paths += glob.glob(os.path.join(sp, "nvidia", "nccl", "lib", "libnccl.so.2*"))

if not _nccl_paths:
    print("  Installing nvidia-nccl-cu12 (latest)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "nvidia-nccl-cu12"],
        check=True
    )
    for sp in site.getsitepackages():
        _nccl_paths += glob.glob(os.path.join(sp, "nvidia", "nccl", "lib", "libnccl.so.2*"))

# Sort descending (newest version name first)
_nccl_paths = sorted(set(_nccl_paths), reverse=True)
_nccl_ok = False
for _p in _nccl_paths:
    try:
        ctypes.CDLL(_p, os.RTLD_GLOBAL)
        print(f"  ✅ Injected: {_p}")
        _nccl_ok = True
        break
    except OSError as e:
        print(f"  ✗ {_p}: {e}")

if not _nccl_ok:
    print("  ⚠️ Could not inject NCCL — restart kernel if engine init fails")

# ── Fix 2: paddlex re-init guard ──────────────────────────────────────────────
for sp in site.getsitepackages():
    core_path = os.path.join(sp, "paddlex", "repo_manager", "core.py")
    if not os.path.isfile(core_path):
        continue
    with open(core_path) as f:
        src = f.read()
    if "return  # Already initialized" in src:
        print("✅ paddlex re-init guard already patched")
        break
    new_src = re.sub(
        r'raise RuntimeError\(\s*["\']PDX has already been initialized[^"\']*["\']\s*\)',
        'return  # Already initialized — skip silently',
        src, count=1
    )
    if new_src != src:
        with open(core_path, "w") as f:
            f.write(new_src)
        print("✅ Patched paddlex re-init guard")
    break

# ── Fix 3: modelscope logger — skip torch import (ALL occurrences) ────────────
# logger.py has >1 call to iutil.find_spec('torch'). Prior sessions may have
# only patched the first one (count=1). We now check for any REMAINING unpatched
# occurrences and replace ALL of them, then delete the .pyc cache so Python
# recompiles from the patched source.
_TORCH_SPEC_PATTERN = r"if iutil\.find_spec\(['\"]torch['\"]\)\s+is not None:"
for sp in site.getsitepackages():
    logger_path = os.path.join(sp, "modelscope", "utils", "logger.py")
    if not os.path.isfile(logger_path):
        continue
    with open(logger_path) as f:
        src = f.read()

    # Check if any unpatched occurrence still exists (NOT just sentinel presence)
    if not re.search(_TORCH_SPEC_PATTERN, src):
        print("✅ modelscope logger already patched (all occurrences)")
    else:
        # Replace ALL occurrences
        new_src = re.sub(_TORCH_SPEC_PATTERN, "if False:  # PATCHED_NCCL", src)
        if new_src == src:
            # Fallback plain replace (both quote styles)
            new_src = src.replace(
                "if iutil.find_spec('torch') is not None:",
                "if False:  # PATCHED_NCCL"
            ).replace(
                'if iutil.find_spec("torch") is not None:',
                "if False:  # PATCHED_NCCL"
            )
        if new_src != src:
            with open(logger_path, "w") as f:
                f.write(new_src)
            count_patched = new_src.count("# PATCHED_NCCL")
            print(f"✅ Patched modelscope logger — {count_patched} occurrence(s) disabled")
        else:
            print("⚠️ modelscope logger: pattern not matched — dumping find_spec lines:")
            for i, ln in enumerate(src.splitlines()):
                if "find_spec" in ln and "torch" in ln:
                    print(f"   line {i+1}: {repr(ln)}")

    # Always delete .pyc cache so Python recompiles from the (now patched) source
    pyc_pattern = os.path.join(sp, "modelscope", "utils", "__pycache__", "logger*.pyc")
    for pyc in glob.glob(pyc_pattern):
        try:
            os.remove(pyc)
            print(f"  Deleted stale pyc: {os.path.basename(pyc)}")
        except OSError:
            pass
    break

# ── Fix 4: Clear stale torch + paddle* + modelscope modules ───────────────────
# torch._C may have left partial state from a prior failed import
_stale = [k for k in sys.modules
          if k.startswith(("torch", "paddlex", "paddleocr", "modelscope"))]
if _stale:
    for mod in _stale:
        del sys.modules[mod]
    print(f"✅ Cleared {len(_stale)} stale module(s)")


class PaddleOCREngine:
    """Wrapper for PaddleOCR 3.x (PP-OCRv5) — identical to preprocess_ocr.py."""

    def __init__(self, device: str = "gpu:0"):
        from paddleocr import PaddleOCR
        self.ocr = PaddleOCR(
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            device=device,
        )
        self.name = "paddleocr_3.x_pp-ocrv5"

    def extract(self, image_path: str) -> dict:
        result = list(self.ocr.predict(image_path))
        texts, confidences = [], []

        for page_result in result:
            try:
                rec_texts = list(page_result["rec_texts"])
                rec_scores = page_result["rec_scores"]
                if hasattr(rec_scores, "tolist"):
                    rec_scores = rec_scores.tolist()
                else:
                    rec_scores = list(rec_scores)
            except (KeyError, TypeError):
                rec_texts, rec_scores = [], []
            texts.extend(rec_texts)
            confidences.extend(rec_scores)

        combined = " ".join(str(t) for t in texts if str(t).strip())
        avg_conf = sum(confidences) / len(confidences) if confidences else 0.0

        return {
            "ocr_text": combined,
            "confidence": round(avg_conf, 4),
            "num_boxes": len(texts),
            "details": [
                {"text": str(t), "score": round(float(s), 4)}
                for t, s in zip(texts, confidences)
            ],
        }


# Initialize engine
engine = PaddleOCREngine(device="gpu:0")
print(f"✅ Using {engine.name} on GPU")


Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.


→ Injecting new NCCL symbols into process...
  ✅ Injected: /usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib/libnccl.so.2
✅ paddlex re-init guard already patched
✅ modelscope logger already patched (all occurrences)
  Deleted stale pyc: logger.cpython-312.pyc
✅ Cleared 695 stale module(s)


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_rec`.


✅ Using paddleocr_3.x_pp-ocrv5 on GPU


## Run OCR on All Images
Auto-resumes from where it left off. Saves per-image JSON to `dataset/img_txt_collab/` and a consolidated mapping. Periodic saves every 500 images to protect against disconnects.

In [5]:
import json
import time
from tqdm.notebook import tqdm

# Get all images
image_paths = sorted(IMG_DIR.glob("*.jpg"))
if not image_paths:
    image_paths = sorted(IMG_DIR.glob("*.png"))
print(f"Total images: {len(image_paths)}")

# Determine already processed (resume support)
already_done = {p.stem for p in OCR_OUTPUT_DIR.glob("*.json")}
to_process = [p for p in image_paths if p.stem not in already_done]

print(f"Already processed: {len(already_done)}")
print(f"Remaining: {len(to_process)}")

# Load existing consolidated data for resume
consolidated = {}
if OCR_CONSOLIDATED.exists():
    with open(OCR_CONSOLIDATED, "r", encoding="utf-8") as f:
        consolidated = json.load(f)

failures = []
start_time = time.time()
SAVE_INTERVAL = 500  # Save consolidated every N images (protect against disconnects)

for i, img_path in enumerate(tqdm(to_process, desc="OCR Extraction")):
    tweet_id = img_path.stem
    try:
        result = engine.extract(str(img_path))
        result["tweet_id"] = tweet_id
        result["ocr_engine"] = engine.name

        # Save per-image JSON
        out_file = OCR_OUTPUT_DIR / f"{tweet_id}.json"
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False)

        consolidated[tweet_id] = result["ocr_text"]

    except Exception as e:
        failures.append(f"{tweet_id}: {e}")

    # Periodic save
    if (i + 1) % SAVE_INTERVAL == 0:
        with open(OCR_CONSOLIDATED, "w", encoding="utf-8") as f:
            json.dump(consolidated, f, ensure_ascii=False)
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(to_process) - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{len(to_process)}] Saved checkpoint | {rate:.1f} img/s | ~{remaining/60:.0f}min left")

# Final save
with open(OCR_CONSOLIDATED, "w", encoding="utf-8") as f:
    json.dump(consolidated, f, ensure_ascii=False)

if failures:
    with open(OCR_FAILURES, "w") as f:
        f.write("\n".join(failures))
    print(f"\n⚠️ {len(failures)} failures logged → {OCR_FAILURES}")

elapsed = time.time() - start_time
print(f"\n✅ Done! Processed {len(to_process)} images in {elapsed:.0f}s ({elapsed/max(len(to_process),1):.3f}s/img)")
print(f"Total in consolidated: {len(consolidated)}")

Total images: 150000
Already processed: 95482
Remaining: 54518


OCR Extraction:   0%|          | 0/54518 [00:00<?, ?it/s]

  [500/54518] Saved checkpoint | 7.0 img/s | ~129min left
  [1000/54518] Saved checkpoint | 6.9 img/s | ~130min left
  [1500/54518] Saved checkpoint | 6.8 img/s | ~129min left
  [2000/54518] Saved checkpoint | 6.8 img/s | ~128min left
  [2500/54518] Saved checkpoint | 6.9 img/s | ~126min left
  [3000/54518] Saved checkpoint | 6.9 img/s | ~125min left
  [3500/54518] Saved checkpoint | 6.9 img/s | ~123min left
  [4000/54518] Saved checkpoint | 6.9 img/s | ~122min left
  [4500/54518] Saved checkpoint | 6.9 img/s | ~121min left
  [5000/54518] Saved checkpoint | 6.9 img/s | ~119min left
  [5500/54518] Saved checkpoint | 6.9 img/s | ~118min left
  [6000/54518] Saved checkpoint | 6.9 img/s | ~118min left
  [6500/54518] Saved checkpoint | 6.9 img/s | ~117min left
  [7000/54518] Saved checkpoint | 6.8 img/s | ~116min left
  [7500/54518] Saved checkpoint | 6.8 img/s | ~115min left
  [8000/54518] Saved checkpoint | 6.9 img/s | ~113min left
  [8500/54518] Saved checkpoint | 6.9 img/s | ~112min lef

: 

: 

## Results Summary
Quick stats on the Colab OCR run.

In [13]:
# Stats on the Colab OCR output
import json

files = list(OCR_OUTPUT_DIR.glob("*.json"))
has_text = 0
no_text = 0

for f in files:
    with open(f, encoding="utf-8") as fp:
        d = json.load(fp)
    if d.get("ocr_text", "").strip():
        has_text += 1
    else:
        no_text += 1

total = has_text + no_text
print(f"Total OCR files in img_txt_collab: {total}")
print(f"  Has text: {has_text} ({100*has_text/max(total,1):.1f}%)")
print(f"  Empty:    {no_text} ({100*no_text/max(total,1):.1f}%)")

# Compare with local run if available
local_dir = DRIVE_PROJECT_ROOT / "dataset" / "img_txt_new"
if local_dir.exists():
    local_count = len(list(local_dir.glob("*.json")))
    print(f"\nLocal img_txt_new: {local_count} files")
    print(f"Colab img_txt_collab: {total} files")
    print(f"Combined coverage: {local_count + total} (may overlap)")

Total OCR files in img_txt_collab: 108898
  Has text: 60406 (55.5%)
  Empty:    48492 (44.5%)
